In [1]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Admin\AppData\Roaming\uv\python\cpython-3.12.8-windows-x86_64-none\Lib\importlib\__init__.py", line 90, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootst

C:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\


In [2]:
import pandas as pd

old_df = pd.read_csv("../../data/labels/Overlaps_1s.csv")

In [3]:
pd.set_option('display.max_columns', None)
old_abs_df = old_df[old_df["Abs"] == 1]
old_abs_df["Clip_Number"].value_counts(dropna=False)
old_abs_df["Origin"].value_counts(dropna=False)

Origin
3s_wavs        6667
1s_absences     202
Name: count, dtype: int64

In [4]:
old_abs_df.loc[old_abs_df["Origin"] == "1s_absences", "Clip_Number"] = 0

old_abs_df["start_s"] = old_abs_df["Clip_Number"].astype(int) - 1
old_abs_df["end_s"] = old_abs_df["start_s"] + 1

In [5]:
old_abs_df["start_s"].value_counts(dropna=False)


start_s
 2    2244
 0    2218
 1    2205
-1     202
Name: count, dtype: int64

In [6]:
old_abs_df.rename(columns={"Timestamp": "snippet_start_time"}, inplace=True)
old_abs_df["snippet_start_time"] = pd.to_datetime(old_abs_df["snippet_start_time"], format='mixed')

In [7]:
old_abs_df.head(2)

,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Site,snippet_start_time,SnippetFilename,Begin File,HydrophoneModel,HydrophoneSensitivity,Clip_Number,Overlaps_str,ClipFilename,Origin,ECHO,HFPC,CC,Mixed,Whistle,Unsure,Abs,Skipped,Boat,N_HF_Call_Types,start_s,end_s
22,56.0,56.0,56.0,56.0,56.0,56.0,BSM,2017-07-24 13:13:25.590,BSM_20170724_13132559.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13132559_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3
27,61.0,61.0,61.0,61.0,61.0,61.0,BSM,2017-07-24 13:21:50.000,BSM_20170724_13215000.wav,NaN,201359382,-172.7,2.0,a,BSM_20170724_13215000_s2,3s_wavs,0,0,0,0,0,0,1,0,0,0,1,2


In [8]:
old_abs_df["clip_start_time"] = old_abs_df["snippet_start_time"] + pd.to_timedelta(old_abs_df["Clip_Number"], unit="s")
old_abs_df["clip_end_time"] = old_abs_df["clip_start_time"] + pd.to_timedelta(1, unit="s")

In [9]:
old_abs_df["clip_filename"] = old_abs_df["Site"] + "_" + old_abs_df["clip_start_time"].dt.strftime("%Y%m%d_%H%M%S%f").str[:-4] + ".wav"
old_abs_df


,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Site,snippet_start_time,SnippetFilename,Begin File,HydrophoneModel,HydrophoneSensitivity,Clip_Number,Overlaps_str,ClipFilename,Origin,ECHO,HFPC,CC,Mixed,Whistle,Unsure,Abs,Skipped,Boat,N_HF_Call_Types,start_s,end_s,clip_start_time,clip_end_time,clip_filename
22,56.0,56.0,56.0,56.0,56.0,56.0,BSM,2017-07-24 13:13:25.590,BSM_20170724_13132559.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13132559_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:13:28.590,2017-07-24 13:13:29.590,BSM_20170724_13132859.wav
27,61.0,61.0,61.0,61.0,61.0,61.0,BSM,2017-07-24 13:21:50.000,BSM_20170724_13215000.wav,NaN,201359382,-172.7,2.0,a,BSM_20170724_13215000_s2,3s_wavs,0,0,0,0,0,0,1,0,0,0,1,2,2017-07-24 13:21:52.000,2017-07-24 13:21:53.000,BSM_20170724_13215200.wav
28,61.0,61.0,61.0,61.0,61.0,61.0,BSM,2017-07-24 13:21:50.000,BSM_20170724_13215000.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13215000_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:21:53.000,2017-07-24 13:21:54.000,BSM_20170724_13215300.wav
31,62.0,62.0,62.0,62.0,62.0,62.0,BSM,2017-07-24 13:22:00.500,BSM_20170724_13220050.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13220050_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:22:03.500,2017-07-24 13:22:04.500,BSM_20170724_13220350.wav
40,68.0,68.0,68.0,68.0,68.0,68.0,BSM,2017-07-24 13:26:06.090,BSM_20170724_13260609.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13260609_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:26:09.090,2017-07-24 13:26:10.090,BSM_20170724_13260909.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10928,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:24:23.000,CAC_20210714_09242300.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09242300,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:24:23.000,2021-07-14 09:24:24.000,CAC_20210714_09242300.wav
10929,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:24:28.000,CAC_20210714_09242800.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09242800,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:24:28.000,2021-07-14 09:24:29.000,CAC_20210714_09242800.wav
10930,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:25:16.000,CAC_20210714_09251600.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09251600,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:25:16.000,2021-07-14 09:25:17.000,CAC_20210714_09251600.wav
10931,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:27:18.000,CAC_20210714_09271800.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09271800,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:27:18.000,2021-07-14 09:27:19.000,CAC_20210714_09271800.wav


In [10]:
import os
import librosa
import soundfile as sf
from tqdm import tqdm

# Output directory
output_dir = "../../data/Verified_Dataset/clip_wavs/"
os.makedirs(output_dir, exist_ok=True)

snippets_dir = "../../data/Full_Dataset/Snippets_3s_wav/"

old_abs_df_3s = old_abs_df[old_abs_df["Origin"] == "3s_wavs"]

grouped = old_abs_df_3s.groupby(["SnippetFilename"])

for (snippet_filename, ), group in tqdm(grouped, total=len(grouped)):
    # print(snippet_filename)
    source_path = os.path.join(snippets_dir, snippet_filename)
    try:
        # Load the entire snippet once
        y, sr = librosa.load(source_path, sr=None)
        
        # Extract all clips from this snippet
        for idx, row in group.iterrows():
            output_path = os.path.join(output_dir, row["clip_filename"])
            
            # Skip if already exists
            if os.path.exists(output_path):
                continue
            
            # Calculate sample indices
            start_sample = int(row["start_s"] * sr)
            end_sample = int(row["end_s"] * sr)
            
            # Extract and save the clip
            clip = y[start_sample:end_sample]
            sf.write(output_path, clip, sr)
            
    except Exception as e:
        print(f"Error processing {snippet_filename}: {e}")


  0%|          | 0/2338 [00:00<?, ?it/s]c:\Users\Admin\Desktop\Emmanuel\beluga-call-pipeline\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 2338/2338 [00:22<00:00, 106.26it/s]


In [11]:
old_abs_df

,Unnamed: 0.5,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Site,snippet_start_time,SnippetFilename,Begin File,HydrophoneModel,HydrophoneSensitivity,Clip_Number,Overlaps_str,ClipFilename,Origin,ECHO,HFPC,CC,Mixed,Whistle,Unsure,Abs,Skipped,Boat,N_HF_Call_Types,start_s,end_s,clip_start_time,clip_end_time,clip_filename
22,56.0,56.0,56.0,56.0,56.0,56.0,BSM,2017-07-24 13:13:25.590,BSM_20170724_13132559.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13132559_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:13:28.590,2017-07-24 13:13:29.590,BSM_20170724_13132859.wav
27,61.0,61.0,61.0,61.0,61.0,61.0,BSM,2017-07-24 13:21:50.000,BSM_20170724_13215000.wav,NaN,201359382,-172.7,2.0,a,BSM_20170724_13215000_s2,3s_wavs,0,0,0,0,0,0,1,0,0,0,1,2,2017-07-24 13:21:52.000,2017-07-24 13:21:53.000,BSM_20170724_13215200.wav
28,61.0,61.0,61.0,61.0,61.0,61.0,BSM,2017-07-24 13:21:50.000,BSM_20170724_13215000.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13215000_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:21:53.000,2017-07-24 13:21:54.000,BSM_20170724_13215300.wav
31,62.0,62.0,62.0,62.0,62.0,62.0,BSM,2017-07-24 13:22:00.500,BSM_20170724_13220050.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13220050_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:22:03.500,2017-07-24 13:22:04.500,BSM_20170724_13220350.wav
40,68.0,68.0,68.0,68.0,68.0,68.0,BSM,2017-07-24 13:26:06.090,BSM_20170724_13260609.wav,NaN,201359382,-172.7,3.0,a,BSM_20170724_13260609_s3,3s_wavs,0,0,0,0,0,0,1,0,0,0,2,3,2017-07-24 13:26:09.090,2017-07-24 13:26:10.090,BSM_20170724_13260909.wav
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10928,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:24:23.000,CAC_20210714_09242300.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09242300,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:24:23.000,2021-07-14 09:24:24.000,CAC_20210714_09242300.wav
10929,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:24:28.000,CAC_20210714_09242800.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09242800,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:24:28.000,2021-07-14 09:24:29.000,CAC_20210714_09242800.wav
10930,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:25:16.000,CAC_20210714_09251600.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09251600,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:25:16.000,2021-07-14 09:25:17.000,CAC_20210714_09251600.wav
10931,NaN,NaN,NaN,NaN,NaN,NaN,CAC,2021-07-14 09:27:18.000,CAC_20210714_09271800.wav,201359382.210714075958.wav,201359382,-172.7,0.0,a,CAC_20210714_09271800,1s_absences,0,0,0,0,0,0,1,0,0,0,-1,0,2021-07-14 09:27:18.000,2021-07-14 09:27:19.000,CAC_20210714_09271800.wav


In [12]:
old_abs_df_1s = old_abs_df[old_abs_df["Origin"] == "1s_absences"]
old_abs_df_1s[["ClipFilename", "Origin", "clip_filename"]]

,ClipFilename,Origin,clip_filename
10731,CAC_20210714_08071100,1s_absences,CAC_20210714_08071100.wav
10732,CAC_20210714_08110800,1s_absences,CAC_20210714_08110800.wav
10733,CAC_20210714_08114800,1s_absences,CAC_20210714_08114800.wav
10734,CAC_20210714_08123100,1s_absences,CAC_20210714_08123100.wav
10735,CAC_20210714_08131200,1s_absences,CAC_20210714_08131200.wav
...,...,...,...
10928,CAC_20210714_09242300,1s_absences,CAC_20210714_09242300.wav
10929,CAC_20210714_09242800,1s_absences,CAC_20210714_09242800.wav
10930,CAC_20210714_09251600,1s_absences,CAC_20210714_09251600.wav
10931,CAC_20210714_09271800,1s_absences,CAC_20210714_09271800.wav


In [13]:
import shutil

new_absences_dir = "../../data/Full_Dataset/New_Absences_1s"
test_dir = "../../data/Verified_Dataset/test/"
for filename in os.listdir(new_absences_dir):
    if filename.endswith(".wav"):
        src_path = os.path.join(new_absences_dir, filename)
        dest_path = os.path.join(output_dir, filename)
        if filename == "CAC_20210714_08110900.wav":
            print(dest_path)
        if not os.path.exists(dest_path):
            shutil.copy2(src_path, dest_path)


In [14]:
save_df = old_abs_df[["Site", "clip_start_time", "clip_end_time", "clip_filename", "start_s", "end_s", "snippet_start_time", "SnippetFilename", "Origin", "Begin File", "HydrophoneSensitivity", "HydrophoneModel"]]

In [15]:
save_df.rename(columns={"Begin File": "original_filename"}, inplace=True)

save_df["ECHO"] = 0
save_df["BBPC"] = 0
save_df["HFPC"] = 0
save_df["Whistle"] = 0
save_df["Boat"] = pd.NA


In [16]:
save_df.to_csv("../../data/Verified_Dataset/labels/labels_old_abs.csv", index=False)